# Importing Matplotlib figures

`Canvas.from_matplotlib` turns a finished Matplotlib figure into a canvas, so a figure
from another library or an old script can be drawn again with the TikZ, Plotly or
plotext backend. The [Matplotlib import](/maxplotlib/guides/matplotlib-import/) guide lists
everything that is imported; this tutorial shows the workflow.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from maxplotlib import Canvas

## A figure made with Matplotlib

Two panels: lines, colored markers and a legend on the left, an image with a color
bar and an annotation on the right.

In [ ]:
x = np.linspace(0, 2 * np.pi, 60)
fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.4))
left.plot(x, np.sin(x), label="sin")
left.scatter(x[::6], np.cos(x[::6]), c=x[::6], label="cos samples")
left.set(title="Lines and markers", xlabel="x")
left.legend()

field = np.outer(np.sin(x), np.cos(x))
image = right.imshow(field, origin="lower", cmap="RdBu_r")
fig.colorbar(image, ax=right, label="field")
right.annotate("peak", (15, 0), xytext=(35, 20), arrowprops={"arrowstyle": "->"})
right.set_title("Image")
plt.close(fig)  # shown below through the canvas

## Importing it

The canvas keeps the layout, the panels and every artist. With the Matplotlib
backend it looks like the original.

In [ ]:
canvas = Canvas.from_matplotlib(fig)
fig_imported, axes = canvas.show()

## What was imported

`canvas.import_report` lists one diagnostic per artist: how it was imported
(`fallback`), and which backends can draw it (`backends`).

In [ ]:
for item in canvas.import_report.diagnostics:
    print(
        f"{item.artist_type:15} {item.severity:5} fallback={item.fallback}: {item.message}"
    )

## Other backends

Lines, markers and images become plot entries that every backend draws. Some
Matplotlib features are kept as native Matplotlib geometry instead, here the
color bar, and a backend that cannot draw them refuses with a clear error:

In [ ]:
try:
    canvas.show(backend="plotly")
except NotImplementedError as error:
    print(error)

`fallback="raster"` flattens the whole figure into one image, which every
backend shows; `fallback="skip"` leaves the unsupported artists out with a warning.

In [ ]:
flat = Canvas.from_matplotlib(fig, fallback="raster")
flat.show(backend="plotly")

## Importing one panel

An `Axes` (or an array of them) imports just those panels. The left panel has only
portable artists, so it converts to TikZ/pgfplots code for a paper:

In [ ]:
panel = Canvas.from_matplotlib(left, width="8cm", ratio=0.6)
tikz = panel.render(backend="tikzfigure")
print(str(tikz)[:600])

In [ ]:
panel.show(backend="tikzfigure")

## Strict imports and files

`strict=True` raises instead of warning when something cannot be imported, for
scripts that must not lose anything silently. `from_matplotlib` also reads a
pickled figure from a path or bytes, but only with `trusted=True`: unpickling can
run arbitrary code, so read only files you made yourself.

In [ ]:
strict = Canvas.from_matplotlib(left, strict=True)
print(len(strict.import_report.diagnostics), "artists imported")